# Questão 1
## 1. Motivação
Como evoluiu a proporção de candidaturas femininas desde a obrigatoriedade do preenchimento da política de cotas?

Em 1995, a Lei nº 9.100/95 previu a reserva mínima de 20% de candidaturas para mulheres nas eleições municipais de 1996. No ano seguinte, a  Lei nº 9.504/1997 (Lei das Eleições) ampliou a medida de forma permanente, para as eleições proporcionais (deputados federais, estaduais/distritais e vereadores) em todo o país.
Alguns anos depois, em 2009, a Lei nº 12.034/2009, fixou que cada partido ou coligação preencheria o mínimo de 30% e o máximo de 70% do número de vagas para candidaturas de cada sexo, modelo que segue até os dias atuais.
Além disso, a Constituição Federal determina que os partidos devem destinar, no mínimo, 30% dos recursos de campanha e do tempo de rádio e TV às mulheres, na mesma proporção do número de candidatas. #confirmar dados

## 2. Hipótese
Supomos que a proporção de candidatas cresce ao longo do tempo e se aproxima do mínimo legal de 30% nos cargos proporcionais, mas não nos majoritários.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"

cols = ["ano", "tipo_eleicao", "cargo", "genero", "situacao"]
df = pd.read_parquet("../data/candidatos.parquet", columns=cols)
df = df[df["ano"] <= 2024].copy()
df.shape

(2925021, 5)

In [2]:
for c in ["tipo_eleicao", "cargo", "genero", "situacao"]:
    print(df[c].value_counts(dropna=False).head(10), "\n")

print(sorted(df["ano"].unique()))
pd.crosstab(df["ano"], df["cargo"])   # quais cargos existem em quais anos

tipo_eleicao
eleicao ordinaria                       2920755
eleicoes suplementares 2008                 619
eleicoes municipais 2020 - ap               549
senador                                      39
eleicao suplementar senador - mt             35
eleicao suplementar - alto santo/ce          33
eleicao suplementar de boqueirao/pb          32
eleicoes suplementares tacaimbo 2023         26
rj - suplementar de silva jardim             20
suplementar gilbues-pi                       20
Name: count, dtype: int64 

cargo
vereador              2578787
vice-prefeito          102076
prefeito               100492
deputado estadual       89572
deputado federal        41682
deputado distrital       4804
senador                  1598
vice-governador          1234
governador               1184
2º suplente               901
Name: count, dtype: int64 

genero
masculino    2064102
feminino      858905
None            2014
Name: count, dtype: int64 

situacao
deferido                  2730696
inde

cargo,1º suplente,1º suplente senador,2º suplente,2º suplente senador,deputado distrital,deputado estadual,deputado federal,governador,prefeito,presidente,senador,vereador,vice-governador,vice-prefeito,vice-presidente
ano,,,,,,,,,,,,,,,
2002,0,313,0,312,663,11628,4356,197,0,6,313,0,196,0,6
2004,0,0,0,0,0,0,0,0,16238,0,0,368105,0,16513,0
2006,0,230,0,233,645,12196,5251,202,0,8,216,0,214,0,8
2008,0,0,0,0,0,0,0,0,15890,0,0,349252,0,16108,0
2010,0,295,0,307,884,14265,5940,168,0,9,268,0,178,0,9
2012,0,0,0,0,0,0,0,0,16114,0,0,449270,0,16467,0
2014,209,0,213,0,1025,16953,7074,191,0,12,182,0,199,0,11
2016,0,0,0,0,0,0,0,0,17124,0,0,462722,0,17418,0
2018,406,0,419,0,981,17916,8567,203,0,14,380,0,210,0,13


## 3. Decisões sobre a análise
Como a política de cotas vale para listas de candidatos, analisamos separadamente os cargos que possuem lista (proporcionais) e os que não possuem (majoritários). A análise foi feita sobre eleições ordinárias e foi dividida entre eleições gerais e municipais. 
Os cargos de vices e suplentes concorrem na chapa do titular e não compõem a lista que a cota regula, além de terem rótulos inconsistentes na base; por isso foram excluídos.
As 2017 candidaturas sem gênero foram excluídas, pois representavam um percentual pequeno dos dados.

### 3.1 Dados de 'situacao'

In [3]:
df[df["situacao"].isna()].groupby("ano").size()

ano
2024    13076
dtype: int64

In [4]:
d24 = df[df["ano"] == 2024]
print("Proporção de situação nula em 2024:", d24["situacao"].isna().mean() * 100, "\n")

nulos = d24[d24["situacao"].isna()]
print("Número de nulos por cargo:\n", nulos["cargo"].value_counts(), "\n")
print("Proporção de gênero entre os nulos:\n", nulos["genero"].value_counts(normalize=True), "\n")
print("Proporção de gênero entre o total:\n", d24["genero"].value_counts(normalize=True), "\n")

Proporção de situação nula em 2024: 2.8252823445781416 

Número de nulos por cargo:
 cargo
vereador         12209
vice-prefeito      505
prefeito           362
Name: count, dtype: int64 

Proporção de gênero entre os nulos:
 genero
masculino    0.630832
feminino     0.369168
Name: proportion, dtype: float64 

Proporção de gênero entre o total:
 genero
masculino    0.656853
feminino     0.343147
Name: proportion, dtype: float64 



As candidaturas com situação não preenchida na base são todas do ano de 2024, sendo 2,8% do total de candidaturas do ano. Como a proporção feminina entre elas é próxima à do restante da eleição, mantê-las ou excluí-las altera a proporção de candidatas em menos de 0,1 ponto percentual, então as mantivemos.

### 3.2 Definição do total de candidaturas

Para decidir se consideraríamos candidaturas inválidas no total, comparamos duas versões: 
- Versão 1, todas as registradas: conta toda candidatura que aparece na base, mesmo as que depois foram indeferidas, renunciaram ou foram canceladas.
- Versão 2, só as válidas: tira as indeferidas, renunciadas, canceladas etc., e conta só as que seguiram. 

Para a maioria das combinações de ano e cargo (39 de 48), a diferença entre as duas versões foi inferior a 1 ponto percentual. As exceções são os cargos majoritários com poucas candidaturas (presidente, senador), em que poucos registros alteram o percentual, e as eleições de 2010 e 2014 para deputado federal e estadual.
Em 2010 e 2014, as mulheres respondem por 35,1% e 48,5% dos registros indeferidos de deputado federal e estadual, contra cerca de 23% e 32% das candidaturas totais nesses cargos. Por isso, a definição do total altera a proporção feminina em 2 a 3 pontos percentuais nesses anos. A base não informa o motivo do indeferimento, de modo que não é possível distinguir suas causas; retomamos o ponto na análise da pergunta 5.

In [5]:
base = df[df["tipo_eleicao"] == "eleicao ordinaria"].copy()
base = base[base["genero"].isin(["feminino", "masculino"])]

proporcionais = ["vereador", "deputado estadual", "deputado federal", "deputado distrital"]
majoritarios  = ["prefeito", "governador", "senador", "presidente"]
base = base[base["cargo"].isin(proporcionais + majoritarios)]

base["tipo_cargo"] = base["cargo"].map(lambda c: "Proporcional" if c in proporcionais else "Majoritário")
base["eleicao"]    = base["cargo"].map(lambda c: "Municipais" if c in ["vereador", "prefeito"] else "Gerais")

invalidas = ["indeferido", "indeferido com recurso", "renuncia", "cancelado",
             "cancelado com recurso", "cassado", "cassado com recurso",
             "falecido", "falecimento", "inelegivel", "impugnado",
             "nao conhecimento do pedido", "pedido nao conhecido",
             "pedido nao conhecido com recurso"]
base["valida"] = ~base["situacao"].isin(invalidas)

In [6]:
def pct_fem(d, chaves):
    t = d.groupby(chaves + ["genero"], observed=True).size().unstack("genero", fill_value=0)
    t["total"] = t["feminino"] + t["masculino"]
    t["pct_fem"] = t["feminino"] / t["total"] * 100
    return t.reset_index()

serie_todas   = pct_fem(base, ["ano", "cargo"])
serie_validas = pct_fem(base[base["valida"]], ["ano", "cargo"])

comp = serie_todas.merge(serie_validas, on=["ano", "cargo"], suffixes=("_todas", "_validas"))
comp["dif_pp"] = comp["pct_fem_validas"] - comp["pct_fem_todas"]
comp["dif_pp"].describe()

count    48.000000
mean      0.029946
std       1.344988
min      -3.067321
25%      -0.414632
50%       0.002623
75%       0.262663
max       5.594406
Name: dif_pp, dtype: float64

In [7]:
cols = ["ano", "cargo", "total_todas", "total_validas",
        "pct_fem_todas", "pct_fem_validas", "dif_pp"]
comp.loc[comp["dif_pp"].abs() > 1, cols].sort_values("dif_pp")

genero,ano,cargo,total_todas,total_validas,pct_fem_todas,pct_fem_validas,dif_pp
18,2010,deputado federal,5940,4888,22.154882,19.087561,-3.067321
25,2014,deputado estadual,16945,14291,31.389790,29.053250,-2.336540
26,2014,deputado federal,7072,5868,31.674208,29.362645,-2.311563
17,2010,deputado estadual,14265,11811,22.754995,20.675641,-2.079353
36,2018,presidente,14,13,14.285714,15.384615,1.098901
29,2014,senador,182,164,18.681319,20.731707,2.050389
28,2014,presidente,12,11,25.000000,27.272727,2.272727
12,2006,presidente,8,7,25.000000,28.571429,3.571429
44,2022,presidente,13,11,30.769231,36.363636,5.594406


In [8]:
sub = base[base["ano"].isin([2010, 2014]) & base["cargo"].isin(["deputado federal", "deputado estadual"])]
inv = sub[~sub["valida"]]
tab = pd.crosstab([inv["ano"], inv["situacao"]], inv["genero"])
tab["total"] = tab.sum(axis=1)
tab["pct_fem"] = (tab["feminino"] / tab["total"] * 100).round(1)
tab.sort_values(["ano", "total"], ascending=[True, False])

genero                           feminino  masculino  total  pct_fem
ano  situacao                                                       
2010 indeferido                       922       1707   2629     35.1
     renuncia                         246        541    787     31.3
     indeferido com recurso             5         47     52      9.6
     cancelado                          6         14     20     30.0
     nao conhecimento do pedido         6          5     11     54.5
     falecido                           1          3      4     25.0
     cassado                            1          1      2     50.0
     cassado com recurso                0          1      1      0.0
2014 indeferido                      1246       1323   2569     48.5
     renuncia                         411        787   1198     34.3
     nao conhecimento do pedido        23         11     34     67.6
     cancelado                          0         25     25      0.0
     indeferido com recurso             4         17     21     19.0
     falecido                           0          7      7      0.0
     cassado                            0          2      2      0.0
     cassado com recurso                0          2      2      0.0

Indeferimentos e renúncias somam 97% das candidaturas retiradas em cada ano, e o desvio feminino está no indeferimento (922 de 2.629 em 2010 e 1.246 de 2.569 em 2014).

## 4. Gráfico

In [9]:
long = comp.melt(id_vars=["ano", "cargo", "total_todas"],
                 value_vars=["pct_fem_todas", "pct_fem_validas"],
                 var_name="definicao", value_name="pct_fem")
long["definicao"] = long["definicao"].map({"pct_fem_todas": "Todas as registradas",
                                           "pct_fem_validas": "Só válidas"})
prop = long[long["cargo"].isin(proporcionais)].copy()
prop["eleicao"] = prop["cargo"].map(lambda c: "Municipais" if c == "vereador" else "Gerais")

In [10]:
%pip install plotly

Note: you may need to restart the kernel to use updated packages.


In [11]:
prop = prop.copy()
prop["ano"] = prop["ano"].astype(int)
prop["cargo"] = prop["cargo"].astype(str)
prop["pct_fem"] = prop["pct_fem"].astype(float)
prop["total_todas"] = prop["total_todas"].astype(int)

print(prop.dtypes)
print(prop.shape, prop["pct_fem"].isna().sum())

ano              int64
cargo           object
total_todas      int64
definicao       object
pct_fem        float64
eleicao         object
dtype: object
(48, 6) 0


In [12]:
import plotly.express as px

p1 = prop[prop["definicao"] == "Todas as registradas"]

fig = px.line(
    p1, x="ano", y="pct_fem", color="cargo", markers=True,
    facet_col="eleicao",
    category_orders={"eleicao": ["Gerais", "Municipais"]},
    hover_data={"total_todas": True},
    labels={"pct_fem": "% de candidaturas femininas", "ano": "Ano da eleição",
            "cargo": "Cargo", "total_todas": "Nº de candidaturas"},
    title="Percentual de candidaturas femininas para cargos proporcionais",
)
fig.update_xaxes(matches=None)
fig.update_xaxes(tickmode="linear", dtick=4, tick0=2002, col=1)
fig.update_xaxes(tickmode="linear", dtick=4, tick0=2004, col=2)
fig.update_yaxes(range=[0, 40], ticksuffix="%")
fig.add_hline(y=30, line_dash="dot", annotation_text="Mínimo legal (30%)",
              annotation_position="bottom right", row="all", col="all")
fig.add_vline(x=2010, line_dash="dash", line_color="gray", col=1)   # 1ª eleição geral após a Lei 12.034/2009
fig.add_vline(x=2012, line_dash="dash", line_color="gray", col=2)   # 1ª eleição municipal após a lei
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("eleicao=", "")))
fig.update_layout(height=500, legend_title_text="Cargo")
fig.show()

In [13]:
import plotly.express as px

p2 = prop[prop["cargo"].isin(["deputado federal", "deputado estadual"])]

fig = px.line(
    p2, x="ano", y="pct_fem", color="cargo", markers=True,
    facet_col="eleicao",
    category_orders={"eleicao": ["Gerais"]},
    hover_data={"total_todas": True},
    labels={"pct_fem": "% de candidaturas femininas", "ano": "Ano da eleição",
            "cargo": "Cargo", "total_todas": "Nº de candidaturas"},
    title="SEU TÍTULO AQUI",
)
fig.update_xaxes(matches=None)
fig.update_xaxes(tickmode="linear", dtick=4, tick0=2002, col=1)
fig.update_xaxes(tickmode="linear", dtick=4, tick0=2004, col=2)
fig.update_yaxes(range=[0, 40], ticksuffix="%")
fig.add_hline(y=30, line_dash="dot", annotation_text="Mínimo legal (30%)",
              annotation_position="bottom right", row="all", col="all")
fig.add_vline(x=2010, line_dash="dash", line_color="gray", col=1)   # 1ª eleição geral após a Lei 12.034/2009
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("eleicao=", "")))
fig.update_layout(height=500, legend_title_text="Cargo")
fig.show()

In [14]:
prop[prop["definicao"] == "Todas as registradas"].pivot(index="ano", columns="cargo", values="pct_fem").round(1)

cargo,deputado distrital,deputado estadual,deputado federal,vereador
ano,,,,
2002,20.7,14.6,11.6,NaN
2004,NaN,NaN,NaN,22.0
2006,20.8,14.1,12.7,NaN
2008,NaN,NaN,NaN,22.0
2010,25.3,22.8,22.2,NaN
2012,NaN,NaN,NaN,32.6
2014,30.7,31.4,31.7,NaN
2016,NaN,NaN,NaN,33.1
2018,31.5,32.0,32.3,NaN
